<a href="https://colab.research.google.com/github/Ewanjohndennis/flyrankml/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

This is a **binary classification** problem that feeds into a **scoring and ranking** output.

Here is what that means in plain terms:

- For each page, I want to predict whether it is likely to need attention — yes or no. That is the classification step.
- I then use the model's probability output (a number between 0 and 1) to rank all pages from most urgent to least urgent. That is the scoring and ranking step.

The final output is not just a yes/no answer — it is an ordered list. The reviewer looks at the top of the list first. So classification is the engine, and ranking is the product.

I am not doing pure clustering (I have a target to predict) and not doing regression (I do not need an exact traffic number, just a priority order).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

**What I would predict:** Whether a page is in a declining trend — a binary label (1 = declining, 0 = not declining).

**Where the label comes from:** In the starter dataset, the label is derived from the `trend_direction` column:

```
is_declining_label = (trend_direction == 'down')
```

This is a **proxy label** — it is not a directly observed future outcome. It is a rule applied to a current measurement. The weakness is that it tells us what is happening now, not what will happen next.

**Why this matters:** A stronger label for the capstone would be future-looking:

```
features from the prior 90 days → did impressions drop by more than X% over the next 30 days?
```

That requires the full warehouse data with proper time windows. For now, `trend_direction == 'down'` is the honest starting proxy — and I am naming it as a proxy, not a ground truth.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/Ewanjohndennis/flyrankml/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Show the label distribution
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print('=== Target label distribution ===')
print(df['is_declining_label'].value_counts())
print()
print(f'Declining (label=1): {df["is_declining_label"].sum():,} pages ({df["is_declining_label"].mean():.1%})')
print(f'Not declining (label=0): {(df["is_declining_label"]==0).sum():,} pages ({(df["is_declining_label"]==0).mean():.1%})')

=== Target label distribution ===
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Declining (label=1): 16,262 pages (54.2%)
Not declining (label=0): 13,738 pages (45.8%)


## 3. Success metric

**My primary metric: Precision@50**

This asks: of the top 50 pages the model flags as most urgent, how many are actually declining?

Why this metric and not overall accuracy:
- A reviewer can realistically check 50 pages per week. What matters is whether those 50 are the right ones.
- Overall accuracy would be misleading here because the classes are nearly balanced (~54% declining) — a dumb model that always says 'declining' would look accurate.
- Precision@50 matches how the output is actually used: someone looks at the top of the ranked list, not the whole thing.

**What 'good' looks like:**
- The starter baseline rule scores Precision@50 = 0.240 (~12 correct out of 50)
- The starter random forest scores Precision@50 = 0.740 (~37 correct out of 50)
- My target for the capstone is to match or beat 0.740 with a stronger, future-looking label and proper client-holdout validation.

**Secondary metrics I will also track:** ROC AUC (overall ranking quality) and Average Precision (quality across all threshold cutoffs).

In [2]:
print('=== What Precision@50 means in practice ===')
print()
print('If you pick the top 50 pages to review:')
print(f'  Baseline rule gets right:  {int(0.240 * 50)} out of 50')
print(f'  Random forest gets right:  {int(0.740 * 50)} out of 50')
print()
print('That gap is why ML is worth building here.')

=== What Precision@50 means in practice ===

If you pick the top 50 pages to review:
  Baseline rule gets right:  12 out of 50
  Random forest gets right:  37 out of 50

That gap is why ML is worth building here.


## 4. The unit of analysis, as a real dataframe

**One row = one content page.**

Each row represents a single page from a client's website, with its search and engagement signals aggregated over a 90-day window. The model scores each page independently and outputs a priority rank.

The code below shows exactly what one row looks like — the columns that matter most for this task.

In [3]:
# Show the unit of analysis as a real dataframe slice
cols_of_interest = [
    'content_id',
    'impressions_90d',
    'sessions_90d',
    'avg_position',
    'ctr',
    'content_age_days',
    'trend_direction',
    'is_declining_label'
]

print('=== Unit of analysis: one row = one content page ===')
print()
print(df[cols_of_interest].head(10).to_string(index=False))
print()
print(f'Total pages (rows): {len(df):,}')
print(f'Total signals (columns): {len(df.columns)}')

=== Unit of analysis: one row = one content page ===

          content_id  impressions_90d  sessions_90d  avg_position  ctr  content_age_days trend_direction  is_declining_label
content_304f48230142             3803            17          10.6 0.76               187            down                   1
content_a1fb4e703a9e            15320             9          20.3 0.05               445            down                   1
content_9aa793d4d895            12581            11          36.5 0.09               141            down                   1
content_331d6c4de07b            11751            78           6.2 0.49               463          stable                   0
content_d99b7a2d90ca            19140           145          44.0 0.13               263            down                   1
content_d4084a4bc775             3970             5           8.5 0.03               147            down                   1
content_9a34b442b552               20             1           7.0 0.00 

## 5. Why ML beats a fixed rule here

A fixed rule works by checking one or two conditions — for example:

```
if age > 180 days AND impressions > 500: flag it
```

The problem is that pages that need attention do not all look the same. A page can be:
- young but already losing traffic fast
- old but still performing well (no refresh needed)
- getting plenty of impressions but almost no clicks (CTR problem, not a content age problem)
- ranking on page one but engagement is terrible once people land

No single if-statement can capture all of these combinations. ML learns which *combinations* of signals matter together — not just individual thresholds.

The starter data already shows this clearly: the top feature by importance is `days_with_impressions` (search visibility consistency over time), not content age. A hand-written rule based on age alone would miss the pages that are consistently losing search presence even if they are not old.

**The honest version:** ML does not guarantee a better answer. It learns a better answer *if* the patterns in the training data generalize to new pages. That is exactly what validation on held-out clients is for — checking whether the learned pattern is real or just memorized noise.

In [4]:
print('=== Content age: declining vs stable pages ===')
print()
declining = df[df['is_declining_label'] == 1]['content_age_days']
stable    = df[df['is_declining_label'] == 0]['content_age_days']

print(f'Median age — declining pages: {declining.median():.0f} days')
print(f'Median age — stable pages:    {stable.median():.0f} days')
print()
print('=== Impressions: declining vs stable pages ===')
print()
declining_imp = df[df['is_declining_label'] == 1]['impressions_90d']
stable_imp    = df[df['is_declining_label'] == 0]['impressions_90d']

print(f'Median impressions — declining pages: {declining_imp.median():,.0f}')
print(f'Median impressions — stable pages:    {stable_imp.median():,.0f}')
print()
print('If age alone explained decline, the age gap above would be large.')
print('The impression gap shows that high-traffic pages are also declining — not just old ones.')

=== Content age: declining vs stable pages ===

Median age — declining pages: 216 days
Median age — stable pages:    287 days

=== Impressions: declining vs stable pages ===

Median impressions — declining pages: 961
Median impressions — stable pages:    472

If age alone explained decline, the age gap above would be large.
The impression gap shows that high-traffic pages are also declining — not just old ones.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.